In [1]:
import os
import json
# Load queries
path = os.getcwd()
with open(os.path.join(path, "data_for_git/factscore_bio.jsonl"), "r", encoding="utf-8") as f:
    queries = [json.loads(line) for line in f]

with open(os.path.join(path, "data_for_git/query_retrieval_top4.json"), "r", encoding="utf-8") as f:
    retrieval = json.loads(f.read())


queries = [query["prompt"] for query in queries]
# queries = queries[:5]  # Uncomment for quick testing
retrievals = [[doc for doc in retrieval[x]] for x in queries]

In [2]:
system_prompt = """You are an AI assistant that provides biographical information based ONLY on the provided Wikipedia context.

IMPORTANT RULES:
- Base your answer EXCLUSIVELY on the retrieved documents
- If the retrieved documents do not contain information about the requested person, then you should clearly state that the retrieved documents do not contain information about the requested person.
- Do NOT use your general knowledge or make up information
- Only mention facts that are explicitly stated in the provided context
- You may provide detailed answers if relevant information is available in the retrieved documents
- The retrieved documents are held within the <CONTEXT> tags.
"""


# Prepare prompts as chat messages for vLLM API
full_prompts = []
m = 0
n = None
for query, retrievals in zip(queries[m:n], retrievals[m:n]):
    user_prompt = f"Retrieved context:\n <CONTEXT>"
    for i, retrieval in enumerate(retrievals):
        title = retrieval["title"]
        content = retrieval["contents"]
        user_prompt += f"START OF DOCUMENT {i} WITH TITLE: {title}\n{content}\n"
        user_prompt += f"\n ================END OF DOCUMENT {i} ================"
    user_prompt += f"<CONTEXT>\nBased on the documents above, i now want you to: {query}, ANSWER:"
    full_prompts.append([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ])

In [3]:
# Generation parameters
n_responses = 5

# Clear the output file (start fresh)
output_file = os.path.join(path, "data_for_git/responses.jsonl")
with open(output_file, "w") as f:
    pass  # Just create/clear the file


In [6]:
from openai.types.chat import ChatCompletion
from nltk import tokenize
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Change working directory to long_form_factuality so relative paths in atomic_facts.py work
original_cwd = os.getcwd()
os.chdir(lff_root)

from long_form_factuality.third_party.factscore.atomic_facts import detect_initials, fix_sentence_splitter

# Keep lff_root available for later use if needed
# Restore original working directory after imports
# os.chdir(original_cwd)

# Helper function to temporarily change cwd for atomic_facts operations
def with_lff_cwd(func):
    """Context manager to run a function with cwd set to long_form_factuality"""
    def wrapper(*args, **kwargs):
        os.chdir(lff_root)
        try:
            result = func(*args, **kwargs)
        finally:
            os.chdir(original_cwd)
        return result
    return wrapper


def gnmt_length_penalty(length: int, alpha: float = 0.6) -> float:
    return ((5 + length) ** alpha) / ((5 + 1) ** alpha)

def get_sequence_logprob(output: ChatCompletion):
    """
    Get the log probabilities of a sequence of tokens from a logprobs array.
    """
    logprobs_list = output.choices[0].logprobs
    sequence_logprob = 0
    token_count = 0
    for token_data in logprobs_list.content:
        token_lp = token_data.logprob
        
        # Sum it up manually
        sequence_logprob += token_lp
        token_count += 1
    return sequence_logprob / token_count / gnmt_length_penalty(token_count)


class MappedToken:
    def __init__(self, text, logprob, start_idx, end_idx):
        self.text = text
        self.logprob = logprob
        self.start_idx = start_idx
        self.end_idx = end_idx

def map_tokens_to_text(logprobs_payload):
    """
    Reconstructs the full text and maps each token to its char offsets.
    Assumes logprobs_payload is a list of objects with .token and .logprob
    """
    full_text = ""
    mapped_tokens = []
    
    current_idx = 0
    
    for item in logprobs_payload.content:
        token_str = item.token  # e.g., " The"
        token_logprob = item.logprob
        
        start = current_idx
        end = current_idx + len(token_str)
        
        mapped_tokens.append(MappedToken(token_str, token_logprob, start, end))
        
        full_text += token_str
        current_idx = end
        
    return full_text, mapped_tokens



def split_and_align(full_text):
    """
    Runs your custom splitter but returns (sentence_text, start_char, end_char)
    """
    sentence_spans = [] # List of tuples: (text, start, end)
    
    # We must track our position in the ORIGINAL full_text
    current_char_offset = 0
    paragraphs = [
        para.strip() for para in full_text.split('\n') if para.strip()
    ]
    for paragraph in paragraphs:
        # 1. Run YOUR existing logic
        initials = detect_initials(paragraph)
        curr_sentences = tokenize.sent_tokenize(paragraph)
        curr_sentences = fix_sentence_splitter(curr_sentences, initials)
        
        # 2. Find the offsets for these specific sentences
        # We search inside the paragraph to find where each sentence lives
        para_offset = 0
        for sent in curr_sentences:
            # Find start of sentence within the paragraph
            # (offset allows us to skip past previous sentences to handle duplicates)
            start_in_para = paragraph.find(sent, para_offset)
            
            if start_in_para == -1:
                # Fallback / Error handling if splitter altered text
                raise ValueError(f"Could not find sentence in paragraph: {sent[:20]}...")
            
            end_in_para = start_in_para + len(sent)
            
            # Calculate absolute global offsets
            global_start = current_char_offset + start_in_para
            global_end = current_char_offset + end_in_para
            
            sentence_spans.append({
                "text": sent,
                "start_char": global_start,
                "end_char": global_end,
                "logprobs": [] # Placeholder
            })
            
            # Update local search offset
            para_offset = end_in_para
            
        # Update global offset by paragraph length + newline (if you split on \n)
        current_char_offset += len(paragraph) + 1 # +1 for the stripped '\n'
        
    return sentence_spans


def assign_logprobs_to_sentences(mapped_tokens, sentence_spans):
    
    sent_idx = 0
    
    for token in mapped_tokens:
        if sent_idx >= len(sentence_spans):
            break
            
        current_sent = sentence_spans[sent_idx]
        
        # Check intersection logic
        # Simple heuristic: If the token starts inside the sentence, it belongs to it.
        # (You can make this stricter by checking midpoints if needed)
        if token.start_idx >= current_sent["start_char"] and token.start_idx < current_sent["end_char"]:
            current_sent["logprobs"].append(token.logprob)
        
        # If token starts AFTER the current sentence ends, move to next sentence
        elif token.start_idx >= current_sent["end_char"]:
            sent_idx += 1
            # Check if it fits the NEW sentence (edge case for back-to-back)
            if sent_idx < len(sentence_spans):
                next_sent = sentence_spans[sent_idx]
                if token.start_idx >= next_sent["start_char"] and token.start_idx < next_sent["end_char"]:
                    next_sent["logprobs"].append(token.logprob)
                    
    return sentence_spans


def get_logprobs_for_sentences(output: ChatCompletion):
    """
    Get the log probabilities of a sequence of tokens from a logprobs array.
    """
    full_text, mapped_tokens = map_tokens_to_text(output.choices[0].logprobs)
    sentence_spans = split_and_align(full_text)
    sentence_spans = assign_logprobs_to_sentences(mapped_tokens, sentence_spans)
    return sentence_spans

from openai import OpenAI
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-key"  # vLLM doesn't require auth, but OpenAI client needs a key
)
prompt = "Tell me a story"
generation = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": prompt}],
    logprobs=True
)
example = get_logprobs_for_sentences(generation)
print(example)

from long_form_factuality.third_party.factscore.atomic_facts import AtomicFactGenerator
generator = AtomicFactGenerator("just_sentences")
paragraphs = generation.choices[0].message.content.split("\n")
paragraphs = [
        para.strip() for para in paragraphs if para.strip()
    ]
print(generator.get_atomic_facts_from_paragraph(paragraphs, just_sentences=True))
    

[{'text': "Sure! Here's a little story for you:", 'start_char': 0, 'end_char': 36, 'logprobs': [-0.5597773790359497, -0.8259493708610535, -0.041778452694416046, -0.25295984745025635, -0.00029595286468975246, -1.5527983903884888, -0.053520604968070984, -0.005827698390930891, 0.0, -0.008382139727473259]}, {'text': '---', 'start_char': 37, 'end_char': 40, 'logprobs': [-0.024960163980722427]}, {'text': 'Once upon a time, in a small village nestled between rolling hills and whispering woods, there lived a young girl named Elara.', 'start_char': 41, 'end_char': 167, 'logprobs': [-0.6435665488243103, -5.578839045483619e-05, 0.0, -3.909988299710676e-05, -0.014165681786835194, -0.008794622495770454, -0.012574790045619011, -1.9009199142456055, -0.11804886162281036, -0.03086887113749981, -0.5854179263114929, -0.2014683037996292, -0.03882652148604393, -0.009362836368381977, -2.4020144939422607, -2.9802276912960224e-06, -0.35386279225349426, -0.0007052318542264402, -0.11292766034603119, -0.01821914

In [5]:
from openai import OpenAI
import threading
import json

# Initialize vLLM client (OpenAI-compatible API)
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-key"  # vLLM doesn't require auth, but OpenAI client needs a key
)
max_concurrent_requests = 400  # Limit how many requests are in flight at once
print(f"Generating {n_responses} responses for {len(full_prompts)} prompts...")
print(f"Max concurrent requests: {max_concurrent_requests}")

# Semaphore to limit concurrent requests
request_semaphore = threading.Semaphore(max_concurrent_requests)

# Thread-safe storage for responses
responses_lock = threading.Lock()
responses_dict = {}

def generate_response(prompt_messages, prompt_key, response_idx):
    """Generate a single response in a thread - vLLM handles batching automatically"""
    # Acquire semaphore - blocks if too many requests are active
    request_semaphore.acquire()
    try:
        completion = client.chat.completions.create(
            model="Qwen/Qwen2.5-7B-Instruct",  # Model name doesn't matter for vLLM
            messages=prompt_messages,
            temperature=0.7,
            top_p=0.9,
            max_tokens=1024,
            logprobs=True
        )
        response_text = completion.choices[0].message.content
        
        # Thread-safe append to responses
        with responses_lock:
            if prompt_key not in responses_dict:
                responses_dict[prompt_key] = []
            responses_dict[prompt_key].append({"response": response_text, "logprobs": get_sequence_logprob(completion)})
            
            # Write to file when we have all responses for this prompt
            if len(responses_dict[prompt_key]) == n_responses:
                item = {"prompt": prompt_key, "responses": responses_dict[prompt_key]}
                with open(output_file, "a") as f:
                    f.write(json.dumps(item) + "\n")
    except Exception as e:
        print(f"Error generating response {response_idx} for prompt: {e}")
    finally:
        # Always release semaphore when done (success or failure)
        request_semaphore.release()

# Create all threads at once - semaphore will limit how many run concurrently
all_threads = []
for prompt_messages in full_prompts:
    prompt_key = prompt_messages[1]["content"]  # user message content
    
    # Create threads for n_responses requests per prompt
    for i in range(n_responses):
        thread = threading.Thread(
            target=generate_response,
            args=(prompt_messages, prompt_key, i)
        )
        thread.start()
        all_threads.append(thread)

# Wait for all requests to complete
for thread in all_threads:
    thread.join()

print(f"Saved responses to {output_file}")
print(f"Total prompts processed: {len(full_prompts)}")


Generating 5 responses for 683 prompts...
Max concurrent requests: 400


KeyboardInterrupt: 